## Load and Structure Training Data



In [1]:
import pandas as pd
print("pandas library imported.")

pandas library imported.


In [2]:
df_excel = pd.read_excel('DataSet.xlsx', sheet_name='data train manual')

transformed_data = []

for column in df_excel.columns:
    label = column
    for text_value in df_excel[column].dropna():
        transformed_data.append({'text': text_value, 'label': label})

df_train = pd.DataFrame(transformed_data)

print("Transformed training data (df_train):")
print(df_train.head())
print(f"Shape of df_train: {df_train.shape}")

Transformed training data (df_train):
                                                text    label
0                 everything alex absolutely perfect  positif
1  everything skinny british man perfect thank ev...  positif
2                 hes little secret anymore hes good  positif
3                                      peaceful song  positif
4                         song criminally underrated  positif
Shape of df_train: (250, 2)


## Load Test Data




In [3]:
df_test = pd.read_excel('DataSet.xlsx', sheet_name='data test')

print("Test data (df_test):")
print(df_test.head())
print(f"Shape of df_test: {df_test.shape}")

Test data (df_test):
                                        cleaned_text
0  everything skinny british man perfect thank ev...
1                                 makes easy cry lt3
2                                     makes easy cry
3              rex bombing us content last days love
4          song doesnt needs many views made legends
Shape of df_test: (760, 1)


## Train Manual Classification Model




In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

print("TfidfVectorizer and LogisticRegression imported.")

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train['text'])
y_train = df_train['label']

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print("TF-IDF vectorization complete.")

# Initialize and train Logistic Regression model
manual_classifier = LogisticRegression(max_iter=1000) # Increased max_iter for convergence
manual_classifier.fit(X_train_tfidf, y_train)

print("Logistic Regression model trained successfully.")

TfidfVectorizer and LogisticRegression imported.
Shape of X_train_tfidf: (250, 1083)
TF-IDF vectorization complete.
Logistic Regression model trained successfully.


## Save Manual Model Components



In [5]:
import joblib

# Save the trained manual classification model
joblib.dump(manual_classifier, 'manual_classifier_model.joblib')
print("Manual classification model saved as 'manual_classifier_model.joblib'")

# Save the TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.joblib')
print("TF-IDF vectorizer saved as 'tfidf_vectorizer.joblib'")

Manual classification model saved as 'manual_classifier_model.joblib'
TF-IDF vectorizer saved as 'tfidf_vectorizer.joblib'


## Train LLM-based Classification Model




In [6]:
import torch
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

print("Libraries imported: torch, LabelEncoder, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer")

Libraries imported: torch, LabelEncoder, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer


In [7]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize the training data
tokenized_train = tokenizer(list(df_train['text']), truncation=True, padding=True)

# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(df_train['label'])

# Create a custom dataset class
class TextClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Instantiate the custom dataset
train_dataset = TextClassificationDataset(tokenized_train, y_train_encoded)

print(f"Tokenizer '{model_name}' instantiated.")
print(f"Training data tokenized. First input_ids: {train_dataset[0]['input_ids'][:5]}...")
print(f"Labels encoded. Original labels: {df_train['label'].unique()}, Encoded labels: {label_encoder.classes_}")
print(f"Custom TextClassificationDataset created with {len(train_dataset)} samples.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer 'distilbert-base-uncased' instantiated.
Training data tokenized. First input_ids: tensor([ 101, 2673, 4074, 7078, 3819])...
Labels encoded. Original labels: ['positif' 'negatif' 'netral'], Encoded labels: ['negatif' 'netral' 'positif']
Custom TextClassificationDataset created with 250 samples.


In [9]:
num_labels = len(label_encoder.classes_)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Define TrainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16, # Not directly used in this setup without a validation set
    num_train_epochs=3,
    # evaluation_strategy='epoch', # Removed due to TypeError: unexpected keyword argument
    save_strategy='epoch',
    # load_best_model_at_end=True, # Removed as it depends on evaluation_strategy
    # metric_for_best_model='loss', # Removed as it depends on evaluation_strategy
    report_to='none' # To prevent logging to external services if not desired
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

# Train the model
trainer.train()

print("LLM-based classification model fine-tuning complete.")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


LLM-based classification model fine-tuning complete.


**Reasoning**:
To ensure the trained LLM model can be reused for future predictions without retraining, I need to save both the fine-tuned model and its tokenizer, similar to how the manual classifier components were saved.



In [10]:
import joblib

# Save the fine-tuned LLM model
trainer.save_model('./llm_model_fine_tuned')
print("LLM-based classification model saved to './llm_model_fine_tuned'")

# Save the tokenizer
tokenizer.save_pretrained('./llm_tokenizer')
print("LLM-based tokenizer saved to './llm_tokenizer'")

# Save the label encoder to ensure consistent label mapping during inference
joblib.dump(label_encoder, 'llm_label_encoder.joblib')
print("Label encoder saved as 'llm_label_encoder.joblib'")

LLM-based classification model saved to './llm_model_fine_tuned'
LLM-based tokenizer saved to './llm_tokenizer'
Label encoder saved as 'llm_label_encoder.joblib'


## Predictions




In [11]:
import joblib

# Load the TF-IDF vectorizer
tfidf_vectorizer = joblib.load('tfidf_vectorizer.joblib')
print("TF-IDF vectorizer loaded.")

# Load the manual classification model
manual_classifier = joblib.load('manual_classifier_model.joblib')
print("Manual classification model loaded.")

# Transform the test data using the loaded TF-IDF vectorizer
X_test_tfidf = tfidf_vectorizer.transform(df_test['cleaned_text'])
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

# Predict labels using the manual classifier
df_test['manual_prediction'] = manual_classifier.predict(X_test_tfidf)

print("Manual model predictions generated and added to df_test.")
print(df_test.head())

TF-IDF vectorizer loaded.
Manual classification model loaded.
Shape of X_test_tfidf: (760, 1083)
Manual model predictions generated and added to df_test.
                                        cleaned_text manual_prediction
0  everything skinny british man perfect thank ev...           positif
1                                 makes easy cry lt3           negatif
2                                     makes easy cry           negatif
3              rex bombing us content last days love           positif
4          song doesnt needs many views made legends           negatif


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import joblib

print("Libraries imported: torch, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, joblib")

# Load the saved LLM tokenizer
tokenizer = AutoTokenizer.from_pretrained('./llm_tokenizer')
print("LLM tokenizer loaded from './llm_tokenizer'.")

# Load the saved fine-tuned LLM model
model = AutoModelForSequenceClassification.from_pretrained('./llm_model_fine_tuned')
print("LLM model loaded from './llm_model_fine_tuned'.")

# Load the saved label encoder
label_encoder = joblib.load('llm_label_encoder.joblib')
print("Label encoder loaded from 'llm_label_encoder.joblib'.")

# Tokenize the test data
tokenized_test = tokenizer(list(df_test['cleaned_text']), truncation=True, padding=True, return_tensors="pt")
print("Test data tokenized for LLM.")

# Create a custom dataset class for test data (without labels)
class TestDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings['input_ids'])

# Instantiate the custom test dataset
test_dataset = TestDataset(tokenized_test)
print(f"Custom TestDataset created with {len(test_dataset)} samples.")

# Define TrainingArguments for prediction (minimal setup)
prediction_args = TrainingArguments(
    output_dir='./results_pred',
    per_device_eval_batch_size=16,
    report_to='none', # To prevent logging to external services if not desired
    do_train=False, # Important: disable training
    do_eval=False,  # Important: disable evaluation
    do_predict=True # Enable prediction
)

# Initialize Trainer for prediction
predictor = Trainer(
    model=model,
    args=prediction_args
)
print("Trainer initialized for LLM prediction.")

# Get predictions (logits)
predictions = predictor.predict(test_dataset)
print("LLM predictions generated.")

# Extract logits and convert to predicted class IDs
logits = predictions.predictions
predicted_class_ids = torch.argmax(torch.tensor(logits), axis=1).numpy()

# Inverse transform predicted class IDs to original labels
df_test['llm_prediction'] = label_encoder.inverse_transform(predicted_class_ids)
print("LLM predictions added to df_test.")

print("\nComparison of Manual vs. LLM Predictions:")
print(df_test[['cleaned_text', 'manual_prediction', 'llm_prediction']].head(10))

Libraries imported: torch, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, joblib
LLM tokenizer loaded from './llm_tokenizer'.
LLM model loaded from './llm_model_fine_tuned'.
Label encoder loaded from 'llm_label_encoder.joblib'.
Test data tokenized for LLM.
Custom TestDataset created with 760 samples.
Trainer initialized for LLM prediction.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


LLM predictions generated.
LLM predictions added to df_test.

Comparison of Manual vs. LLM Predictions:
                                        cleaned_text manual_prediction  \
0  everything skinny british man perfect thank ev...           positif   
1                                 makes easy cry lt3           negatif   
2                                     makes easy cry           negatif   
3              rex bombing us content last days love           positif   
4          song doesnt needs many views made legends           negatif   
5  hrefhttpswwwyoutubecomwatchvk9qtjzb1feampt2504...           positif   
6                                 its really anymore           negatif   
7                                     song relatelt3           positif   
8                                  rex therapist lt3           positif   
9                                  love greater pain           positif   

  llm_prediction  
0        positif  
1        negatif  
2        negatif  
3    

In [13]:
print("Final df_test with predictions:")
print(df_test.head(10))

Final df_test with predictions:
                                        cleaned_text manual_prediction  \
0  everything skinny british man perfect thank ev...           positif   
1                                 makes easy cry lt3           negatif   
2                                     makes easy cry           negatif   
3              rex bombing us content last days love           positif   
4          song doesnt needs many views made legends           negatif   
5  hrefhttpswwwyoutubecomwatchvk9qtjzb1feampt2504...           positif   
6                                 its really anymore           negatif   
7                                     song relatelt3           positif   
8                                  rex therapist lt3           positif   
9                                  love greater pain           positif   

  llm_prediction  
0        positif  
1        negatif  
2        negatif  
3        negatif  
4        negatif  
5        negatif  
6        negatif  
7

## Evaluate Model Accuracies

In [15]:
from sklearn.metrics import accuracy_score

# --- Manual Model Training Accuracy ---
# Predict on the training data using the manual classifier
manual_train_predictions = manual_classifier.predict(X_train_tfidf)

# Calculate and print training accuracy
manual_train_accuracy = accuracy_score(y_train, manual_train_predictions)
print(f"Manual Model Training Accuracy: {manual_train_accuracy:.4f}")

# --- LLM Model Training Accuracy ---
# Get predictions (logits) on the training dataset
llm_train_predictions_output = trainer.predict(train_dataset)
logits_train = llm_train_predictions_output.predictions
predicted_class_ids_train = torch.argmax(torch.tensor(logits_train), axis=1).numpy()

# The y_train_encoded already holds the numerical labels for training data
llm_train_accuracy = accuracy_score(y_train_encoded, predicted_class_ids_train)
print(f"LLM Model Training Accuracy: {llm_train_accuracy:.4f}")


Manual Model Training Accuracy: 0.9480


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


LLM Model Training Accuracy: 0.6240


In [14]:
output_file_name = 'predictions_output.xlsx'
df_test.to_excel(output_file_name, index=False)
print(f"Hasil prediksi telah disimpan ke '{output_file_name}'")

Hasil prediksi telah disimpan ke 'predictions_output.xlsx'
